# 08. Graphs and dynamic programming

## Learning goals
1. Traverse graphs with BFS and DFS.
2. Prove Dijkstra correct and state its cost.
3. Order a DAG topologically and build a minimum spanning tree.
4. Solve LIS, 0/1 knapsack and edit distance with dynamic programming.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## BFS and DFS
Both visit each vertex once and scan each edge once: O(V + E).
BFS uses a queue and gives the fewest-edge distance in an unweighted graph. DFS uses a stack and follows one branch to its end.

**BFS proof sketch.** Vertices leave the queue in non-decreasing distance order. A vertex first met from a vertex at distance d gets d+1, and no shorter route exists, or it would have been met earlier.

In [ ]:
from understanding_python.graphs import bfs, dfs
G = {"a": ["b", "c"], "b": ["d"], "c": ["d", "e"], "d": ["f"], "e": ["f"], "f": []}
assert bfs(G, "a") == {"a": 0, "b": 1, "c": 1, "d": 2, "e": 2, "f": 3}
assert dfs(G, "a") == ["a", "b", "d", "f", "c", "e"]
grid = {(r, c): [(r + dr, c + dc) for dr, dc in ((0, 1), (1, 0)) if r + dr < 4 and c + dc < 4] for r in range(4) for c in range(4)}
assert bfs(grid, (0, 0))[(3, 3)] == 6

## Dijkstra
Repeatedly settle the unsettled vertex with the smallest tentative distance, then relax its edges.

**Proof.** Claim: when u is settled, dist[u] is the true shortest distance. Suppose a shorter path existed.
It must leave the settled set through some edge (x, y) with y unsettled. Weights are non-negative, so dist[y] <= the path cost to y <= the path cost to u < dist[u].
Then y would have been chosen before u, a contradiction.

**Cost.** Each edge pushes at most one heap entry: O(E log E) = O((V + E) log V). See `assets/call-graph-dijkstra.svg` for the call graph.

In [ ]:
from understanding_python.graphs import dijkstra, path_to
W = {"s": [("a", 7), ("b", 2)], "a": [("t", 1)], "b": [("a", 3), ("t", 9)], "t": []}
dist, prev = dijkstra(W, "s")
assert dist == {"s": 0, "b": 2, "a": 5, "t": 6}
assert path_to(prev, "s", "t") == ["s", "b", "a", "t"]
try:
    dijkstra({"x": [("y", -1)]}, "x")
    raise AssertionError
except ValueError:
    pass
# cross-check against Bellman-Ford on random graphs
import random
rng = random.Random(9)
for _ in range(25):
    n = 8
    g = {i: [(rng.randrange(n), rng.randint(1, 9)) for _ in range(3)] for i in range(n)}
    d1, _ = dijkstra(g, 0)
    d2 = {0: 0}
    for _ in range(n):
        for u in g:
            for v, w in g[u]:
                if u in d2 and d2[u] + w < d2.get(v, 10**9):
                    d2[v] = d2[u] + w
    assert d1 == d2

## Topological sort
Kahn's algorithm removes vertices with in-degree zero. If every vertex leaves, the order is valid; if not, a cycle remains.
Each vertex and edge is handled once: O(V + E).

In [ ]:
from understanding_python.graphs import topological_sort
deps = {"shirt": ["tie", "belt"], "tie": ["jacket"], "trousers": ["belt", "shoes"], "belt": ["jacket"], "shoes": [], "jacket": []}
order = topological_sort(deps)
pos = {v: i for i, v in enumerate(order)}
assert all(pos[u] < pos[v] for u, vs in deps.items() for v in vs)
try:
    topological_sort({"a": ["b"], "b": ["a"]})
    raise AssertionError
except ValueError as e:
    assert "cycle" in str(e)

## Minimum spanning tree (Kruskal)
Sort edges by weight. Take an edge when it joins two components, checked with union-find.

**Cut property.** For any cut, the lightest crossing edge belongs to some MST. Kruskal adds exactly such an edge at each step.
**Cost.** Sorting dominates: O(E log E).

In [ ]:
from understanding_python.graphs import kruskal
edges = [(0, 1, 4), (0, 2, 3), (1, 2, 1), (1, 3, 2), (2, 3, 4), (3, 4, 2), (4, 5, 6)]
total, used = kruskal(6, edges)
assert total == 14 and len(used) == 5
# brute force: the best spanning tree over all edge subsets of size n-1
from itertools import combinations
from understanding_python.structures import UnionFind
best = min(sum(w for *_, w in c) for c in combinations(edges, 5)
           if (lambda uf: all(uf.union(u, v) for u, v, _ in c))(UnionFind(6)))
assert best == total

## Dynamic programming: LIS
Let best[i] be the longest increasing subsequence ending at i. best[i] = 1 + max best[j] over j < i with a[j] < a[i]. That is O(n^2).

The patience method keeps `tails[k]`, the smallest tail of any increasing subsequence of length k+1. The list stays sorted,
so a binary search places each item: O(n log n). The answer is `len(tails)`.

In [ ]:
from understanding_python.dp import lis_length, lis_quadratic
assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4
rng = random.Random(4)
for _ in range(60):
    a = [rng.randint(0, 30) for _ in range(rng.randint(0, 25))]
    assert lis_length(a) == lis_quadratic(a)

## Dynamic programming: 0/1 knapsack
Let dp[c] be the best value with capacity c. For each item, update c from high to low: dp[c] = max(dp[c], dp[c-w] + v).
Going downward stops an item from being used twice. Time O(n * cap), space O(cap). This is pseudo-polynomial: the input size is log(cap), not cap.

In [ ]:
from understanding_python.dp import knapsack, knapsack_brute
assert knapsack([1, 3, 4, 5], [1, 4, 5, 7], 7) == 9
rng = random.Random(8)
for _ in range(40):
    w = [rng.randint(1, 9) for _ in range(9)]
    v = [rng.randint(1, 30) for _ in range(9)]
    assert knapsack(w, v, 20) == knapsack_brute(w, v, 20)

## Dynamic programming: edit distance
D[i][j] is the cheapest way to turn `s[:i]` into `t[:j]`. Take the minimum of delete, insert and substitute (free when the characters match).
The table has len(s) * len(t) cells with O(1) work each. Keeping two rows cuts space to O(len(t)).
The result is a metric: it is symmetric and obeys the triangle inequality.

In [ ]:
from understanding_python.dp import edit_distance
assert edit_distance("kitten", "sitting") == 3
assert edit_distance("", "") == 0 and edit_distance("abc", "") == 3 and edit_distance("flaw", "lawn") == 2
words = ["graph", "giraffe", "gift", "shaft", "grape"]
for x in words:
    for y in words:
        assert edit_distance(x, y) == edit_distance(y, x)
        for z in words:
            assert edit_distance(x, z) <= edit_distance(x, y) + edit_distance(y, z)

## Benchmark: scaling
Doubling n should roughly double the LIS patience time and quadruple the quadratic table.

In [ ]:
from understanding_python.bench import best_time
rng = random.Random(1)
def data(n): return [rng.randint(0, 10**6) for _ in range(n)]
a1, a2 = data(300), data(600)
q1, q2 = best_time(lambda: lis_quadratic(a1), 1), best_time(lambda: lis_quadratic(a2), 1)
f1, f2 = best_time(lambda: lis_length(a1), 20), best_time(lambda: lis_length(a2), 20)
print(f"quadratic x{q2/q1:.1f} (expect about 4), patience x{f2/f1:.1f} (expect about 2)")
assert q2 / q1 > 2.0
assert best_time(lambda: lis_length(data(3000)), 3) < best_time(lambda: lis_quadratic(data(3000)), 1)